In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
K = 252

**Chargement des prix**

In [ ]:
prices = pd.read_csv("df_train.csv", sep=";", index_col=0)

**Visualitation des données**

In [ ]:
prices.head(10)

In [ ]:
fig, ax = plt.subplots(4,4, figsize=(20,16))
j,k = 0,0
for i,asset in enumerate(prices.columns):
    ax[j,k].plot(prices[asset])
    ax[j,k].set_title(asset)
    if k < 3:
        k+=1
    else:
        k=0
        j+=1

**Quelques fonctions de base**

In [ ]:
def normalize_price(serie):
    '''
    -serie (pd.Serie) : série de prix
    Normalise la série avec le prix à t_0 = 100
    '''
    s = serie/serie.shift(1)
    s.iloc[0] = 100
    return s.cumprod()

def get_returns(df):
    ''' 
    -df (pd.DataFrame ou pd.Series) : dataframe ou série des prix
    Renvoie les returns
    '''
    return (df/df.shift(1)).dropna()

def get_cagr(serie):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    Cumulative Annual Growth Return
    '''
    return round(float(((serie.iloc[-1]/serie.iloc[0])**(1/(len(serie)/252))) - 1), 4)

def get_vol(serie):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    Annualized Volatility
    '''
    return round(float(get_returns(serie).std() * np.sqrt(252)), 4)

def get_sr(serie, r=0.0):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    -r (float) : taux d'intérêt (à 0 pour le projet)
    Annualized Sharpe Ratio
    '''
    return round(float((get_cagr(serie) - r) / get_vol(serie)), 4)

def get_calmar(serie):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    Calmar Ratio
    '''
    return round(get_cagr(serie)/get_mdd(serie), 4)

def get_mdd(serie):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    Max Drawdown
    '''
    running_max = np.maximum.accumulate(serie)
    drawdowns   = (serie - running_max) / running_max
    trough_idx  = int(np.argmin(drawdowns))
    mdd         = -drawdowns.iloc[trough_idx]
    return round(float(mdd), 4)

def get_corr(serie, benchmark):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    -benchmark (pd.Serie) : série de prix du benchmark (généralement S&P500)
    Correlation
    '''
    return round(float(np.corrcoef(get_returns(serie), get_returns(benchmark))[0,1]), 4)

def overall_score(serie, benchmark):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    -benchmark (pd.Serie) : série de prix du benchmark (généralement S&P500)
    Personnalized selection score
    '''
    return round((0.5*get_sr(serie) + 0.5*get_calmar(serie))/get_corr(serie, benchmark),4)

def compute_metrics(serie, benchmark):
    '''
    -serie (pd.Serie) : série de prix (asset ou portfolio)
    -benchmark (pd.Serie) : série de prix du benchmark (généralement S&P500)
    Compute all metrics
    '''
    print(f"CAGR : {round(get_cagr(serie) * 100, 4)} %")
    print(f"Annualized Vol : {round(get_vol(serie)* 100, 4)} %")
    print(f"Sharpe Ratio : {round(get_sr(serie), 4)}")
    print(f"Calmar Ratio : {round(get_calmar(serie), 4)}")
    print(f"Max Drawdown : {round(get_mdd(serie)*100,4)} %")
    print(f"Corr-bench : {round(float(get_corr(serie, benchmark)), 4)}")
    print(f"Overall score : {overall_score(serie, benchmark)}")

#### Fonction à compléter
La fonction doit prendre en entrée un dataframe de taille 252 (moins de données pourront être utilisées dans la fonction si nécessaire) et renvoyer un vecteur (np.array, shape (15,)) de taille 15 comprenant les poids des asset 1 à 15. Il y a deux contraintes sur le vecteur :

- Les poids sont positifs.

- La somme des poids est égale ou inférieure à 1


In [ ]:
def compute_weights(df):
    pass

#### Benchmark Simple - Equally weighted
Une méthode simple de répartition des poids serait de répartir la valeur du portefeuille de manière équipondérée

In [ ]:
eq_weighted_portfolio = normalize_price(((prices*1/15).sum(axis=1)).iloc[K:])

**Fonction permettant de valider les poids**

In [ ]:
def check_weights(w):
    '''
    -w (np.array, shape : (15,)) : Vecteur des poids
    Renvoie False si les poids ne respectent pas les contraintes
    '''
    if round(w.sum(), 3)<=1 and (w<0).sum()==0:
        return True
    else:
        print("Poids négatifs ou somme supérieure à 1")
        print(f"Poids {w}")
        print(f"Somme : {w.sum()}")
        return False

**Chargement des autres benchmark**

In [ ]:
df_benchmark = pd.read_csv("df_benchmark.csv", sep=";", index_col=0)
secret_bench = normalize_price(df_benchmark["Secret"])
sp500 = normalize_price(df_benchmark["SP500"])

**Backtest la stratégie**

In [ ]:
def backtest():
    #Calculate Weights
    weights_t = []
    for i in range(len(prices)-K+1):
        w = compute_weights(prices.iloc[i:i+K])
        if check_weights(w):
            weights_t.append(w)
        else:
            break
    
    #Calculate portfolio value and returns
    pr = (np.array(weights_t)[1:-1] * get_returns(prices).iloc[K:]).sum(axis=1)
    portfolio = [100]
    for i in range(len(pr)):
        portfolio.append(pr.iloc[i]*portfolio[-1])
    portfolio = pd.Series(portfolio)
    portfolio.index = sp500.index

    print("==========PORTFOLIO===========")
    compute_metrics(portfolio, sp500)
    print("==========S&P500==============")
    compute_metrics(sp500, sp500)
    print("==========EQ-WEIGHTED=========")
    compute_metrics(eq_weighted_portfolio, sp500)
    print("======SECRET-BENCHMARK========")
    compute_metrics(secret_bench, sp500)

    plt.figure(figsize=(20,12))
    plt.plot(portfolio)
    plt.plot(sp500)
    plt.plot(eq_weighted_portfolio)
    plt.plot(secret_bench)
    plt.legend(["Portfolio", "S&P500", "Eq-Weighted", "Secret"])
    
    return portfolio

In [ ]:
strat1 = backtest()